<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Описание задачи:
Создать базовый класс Inventory в C#, который будет представлять информацию о
наличии товаров на складе. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.
Требования к базовому классу Inventory:
• Атрибуты: ID склада (WarehouseId), Название склада (WarehouseName),
Общий объем хранения (StorageCapacity).
• Методы:
o
o GetStorageStatus(): метод для получения статуса доступного
пространства на складе.
o AddItem(Item item): метод для добавления товара на склад.
o RemoveItem(Item item): метод для удаления товара со склада.
Требования к производным классам:
1. ПерсональныйСклад (PersonalInventory): Должен содержать
дополнительные атрибуты, такие как Владелец склада (OwnerName).
Метод GetStorageStatus() должен быть переопределен для отображения
информации о владельце склада вместе с статусом хранения.
2. ГрупповойСклад (GroupInventory): Должен содержать дополнительные
атрибуты, такие как Группа товаров (ProductGroup). Метод AddItem() должен
быть переопределен для добавления информации о группе товаров при
добавлении нового товара.
3. АвтоматизированныйСклад (AutomatedInventory) (если требуется третий
класс): Должен содержать дополнительные атрибуты, такие как
Автоматизация уровня (AutomationLevel). Метод RemoveItem() должен быть
переопределен для добавления информации о уровне автоматизации при
удалении товара.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;

// ДЕЛЕГАТЫ
public delegate void InventoryEventHandler(string message);

// БАЗОВЫЙ КЛАСС ДЛЯ ТОВАРА
public class Item
{
    public string ItemId { get; set; }
    public string Name { get; set; }
    public decimal Price { get; set; }
    public int Quantity { get; set; }
    public double Weight { get; set; }
    public string Category { get; set; }
    public string Supplier { get; set; }

    public Item(string itemId, string name, decimal price, int quantity, double weight, string category, string supplier)
    {
        ItemId = itemId;
        Name = name;
        Price = price;
        Quantity = quantity;
        Weight = weight;
        Category = category;
        Supplier = supplier;
    }

    public void DisplayInfo()
    {
        Console.WriteLine($"    Товар: {Name} (ID: {ItemId})");
        Console.WriteLine($"    Цена: {Price} руб., Количество: {Quantity}, Вес: {Weight} кг");
        Console.WriteLine($"    Категория: {Category}, Поставщик: {Supplier}");
    }

    public decimal CalculateTotalValue()
    {
        return Price * Quantity;
    }
}

// БАЗОВЫЙ КЛАСС Inventory
public class Inventory
{
    public string WarehouseId { get; set; }
    public string WarehouseName { get; set; }
    public double StorageCapacity { get; set; }
    public string Location { get; set; }
    public string Manager { get; set; }

    protected List<Item> items;
    protected Dictionary<string, int> categoryCount;

    // СОБЫТИЯ
    public event InventoryEventHandler ItemAdded;
    public event InventoryEventHandler ItemRemoved;

    public Inventory(string warehouseId, string warehouseName, double storageCapacity, string location, string manager)
    {
        WarehouseId = warehouseId;
        WarehouseName = warehouseName;
        StorageCapacity = storageCapacity;
        Location = location;
        Manager = manager;

        items = new List<Item>();
        categoryCount = new Dictionary<string, int>();
    }

    public virtual void GetStorageStatus()
    {
        double usedCapacity = CalculateUsedCapacity();
        double availableCapacity = StorageCapacity - usedCapacity;
        double utilizationPercentage = (usedCapacity / StorageCapacity) * 100;

        Console.WriteLine($"=== СТАТУС СКЛАДА {WarehouseName} ===");
        Console.WriteLine($"ID: {WarehouseId}");
        Console.WriteLine($"Местоположение: {Location}");
        Console.WriteLine($"Менеджер: {Manager}");
        Console.WriteLine($"Общая вместимость: {StorageCapacity} кг");
        Console.WriteLine($"Использовано: {usedCapacity:F2} кг ({utilizationPercentage:F1}%)");
        Console.WriteLine($"Доступно: {availableCapacity:F2} кг");
        Console.WriteLine($"Товаров: {items.Count}, Категорий: {categoryCount.Count}");
    }

    public virtual void AddItem(Item item)
    {
        double neededCapacity = item.Weight * item.Quantity;
        if (CalculateUsedCapacity() + neededCapacity > StorageCapacity)
        {
            Console.WriteLine($"    ОШИБКА: Недостаточно места для {item.Name} (нужно {neededCapacity} кг)");
            return;
        }

        items.Add(item);
        
        if (categoryCount.ContainsKey(item.Category))
            categoryCount[item.Category] += item.Quantity;
        else
            categoryCount[item.Category] = item.Quantity;

        ItemAdded?.Invoke($"{item.Name} добавлен на склад {WarehouseName}");
        Console.WriteLine($"    УСПЕХ: {item.Name} добавлен на склад");
    }

    public virtual void RemoveItem(Item item)
    {
        if (items.Remove(item))
        {
            if (categoryCount.ContainsKey(item.Category))
            {
                categoryCount[item.Category] -= item.Quantity;
                if (categoryCount[item.Category] <= 0)
                    categoryCount.Remove(item.Category);
            }

            ItemRemoved?.Invoke($"{item.Name} удален со склада {WarehouseName}");
            Console.WriteLine($"    УСПЕХ: {item.Name} удален со склада");
        }
        else
        {
            Console.WriteLine($"    ОШИБКА: Товар {item.Name} не найден на складе");
        }
    }

    public void DisplayAllItems()
    {
        Console.WriteLine($"\n=== ТОВАРЫ НА СКЛАДЕ {WarehouseName} ===");
        if (items.Count == 0)
        {
            Console.WriteLine("    Склад пуст");
            return;
        }

        foreach (var item in items)
        {
            item.DisplayInfo();
            Console.WriteLine("    ---");
        }
    }

    public List<Item> FindItemsByCategory(string category)
    {
        return items.Where(item => item.Category.Equals(category, StringComparison.OrdinalIgnoreCase)).ToList();
    }

    public decimal CalculateTotalInventoryValue()
    {
        return items.Sum(item => item.CalculateTotalValue());
    }

    protected double CalculateUsedCapacity()
    {
        return items.Sum(item => item.Weight * item.Quantity);
    }

    public void DisplayCategoryStatistics()
    {
        Console.WriteLine($"\n=== СТАТИСТИКА ПО КАТЕГОРИЯМ ({WarehouseName}) ===");
        if (categoryCount.Count == 0)
        {
            Console.WriteLine("    Нет данных по категориям");
            return;
        }

        foreach (var category in categoryCount)
        {
            Console.WriteLine($"    {category.Key}: {category.Value} единиц");
        }
    }
}

// ПРОИЗВОДНЫЙ КЛАСС 1: PersonalInventory
public class PersonalInventory : Inventory
{
    public string OwnerName { get; set; }
    public string OwnerContact { get; set; }
    public bool IsPrivate { get; set; }
    public int MaxItemsLimit { get; set; }

    private Queue<Item> recentAdditions;

    public PersonalInventory(string warehouseId, string warehouseName, double storageCapacity,
                           string location, string manager, string ownerName, string ownerContact, 
                           bool isPrivate, int maxItemsLimit)
        : base(warehouseId, warehouseName, storageCapacity, location, manager)
    {
        OwnerName = ownerName;
        OwnerContact = ownerContact;
        IsPrivate = isPrivate;
        MaxItemsLimit = maxItemsLimit;
        recentAdditions = new Queue<Item>();
    }

    public override void GetStorageStatus()
    {
        base.GetStorageStatus();
        Console.WriteLine($"Владелец: {OwnerName}");
        Console.WriteLine($"Контакт: {OwnerContact}");
        Console.WriteLine($"Приватный: {(IsPrivate ? "Да" : "Нет")}");
        Console.WriteLine($"Лимит товаров: {items.Count}/{MaxItemsLimit}");
    }

    public override void AddItem(Item item)
    {
        if (items.Count >= MaxItemsLimit)
        {
            Console.WriteLine($"    ОШИБКА: Достигнут лимит {MaxItemsLimit} товаров");
            return;
        }

        base.AddItem(item);
        
        recentAdditions.Enqueue(item);
        if (recentAdditions.Count > 3)
            recentAdditions.Dequeue();
    }

    public void DisplayRecentAdditions()
    {
        Console.WriteLine($"\n=== НЕДАВНО ДОБАВЛЕННЫЕ ТОВАРЫ ({WarehouseName}) ===");
        if (recentAdditions.Count == 0)
        {
            Console.WriteLine("    Нет недавних добавлений");
            return;
        }

        foreach (var item in recentAdditions)
        {
            item.DisplayInfo();
        }
    }

    public void ContactOwner()
    {
        Console.WriteLine($"\nКонтакт владельца {OwnerName}: {OwnerContact}");
    }
}

// ПРОИЗВОДНЫЙ КЛАСС 2: GroupInventory
public class GroupInventory : Inventory
{
    public string ProductGroup { get; set; }
    public string GroupManager { get; set; }
    public decimal GroupBudget { get; set; }

    private Dictionary<string, List<Item>> groupItems;

    public GroupInventory(string warehouseId, string warehouseName, double storageCapacity,
                        string location, string manager, string productGroup, 
                        string groupManager, decimal groupBudget)
        : base(warehouseId, warehouseName, storageCapacity, location, manager)
    {
        ProductGroup = productGroup;
        GroupManager = groupManager;
        GroupBudget = groupBudget;
        groupItems = new Dictionary<string, List<Item>>();
    }

    public override void AddItem(Item item)
    {
        base.AddItem(item);

        if (!groupItems.ContainsKey(item.Category))
            groupItems[item.Category] = new List<Item>();
        
        groupItems[item.Category].Add(item);
        Console.WriteLine($"    Добавлен в группу '{ProductGroup}'");
    }

    public override void GetStorageStatus()
    {
        base.GetStorageStatus();
        Console.WriteLine($"Группа товаров: {ProductGroup}");
        Console.WriteLine($"Менеджер группы: {GroupManager}");
        Console.WriteLine($"Бюджет группы: {GroupBudget:C}");
    }

    public void DisplayGroupCategories()
    {
        Console.WriteLine($"\n=== КАТЕГОРИИ ГРУППЫ '{ProductGroup}' ===");
        if (groupItems.Count == 0)
        {
            Console.WriteLine("    Нет категорий в группе");
            return;
        }

        foreach (var category in groupItems)
        {
            Console.WriteLine($"    {category.Key}: {category.Value.Count} товаров");
        }
    }

    public void AllocateBudget(decimal amount)
    {
        if (amount <= GroupBudget)
        {
            GroupBudget -= amount;
            Console.WriteLine($"\nВыделено {amount:C} из бюджета группы");
            Console.WriteLine($"Остаток бюджета: {GroupBudget:C}");
        }
        else
        {
            Console.WriteLine($"\nНедостаточно средств в бюджете группы");
            Console.WriteLine($"Запрошено: {amount:C}, Доступно: {GroupBudget:C}");
        }
    }
}

// ПРОИЗВОДНЫЙ КЛАСС 3: AutomatedInventory
public class AutomatedInventory : Inventory
{
    public int AutomationLevel { get; set; }
    public bool HasRobotics { get; set; }
    public string SoftwareSystem { get; set; }

    private Queue<string> errorLog;

    public AutomatedInventory(string warehouseId, string warehouseName, double storageCapacity,
                            string location, string manager, int automationLevel, 
                            bool hasRobotics, string softwareSystem)
        : base(warehouseId, warehouseName, storageCapacity, location, manager)
    {
        AutomationLevel = automationLevel;
        HasRobotics = hasRobotics;
        SoftwareSystem = softwareSystem;
        errorLog = new Queue<string>();
    }

    public override void RemoveItem(Item item)
    {
        if (AutomationLevel >= 3)
        {
            Console.WriteLine($"    АВТОМАТИЧЕСКОЕ УДАЛЕНИЕ: Уровень автоматизации {AutomationLevel}");
            Console.WriteLine($"    Используется система: {SoftwareSystem}");
        }

        base.RemoveItem(item);
    }

    public override void GetStorageStatus()
    {
        base.GetStorageStatus();
        Console.WriteLine($"Уровень автоматизации: {AutomationLevel}/5");
        Console.WriteLine($"Робототехника: {(HasRobotics ? "Да" : "Нет")}");
        Console.WriteLine($"Программная система: {SoftwareSystem}");
    }

    public void LogError(string errorMessage)
    {
        errorLog.Enqueue($"{DateTime.Now:HH:mm:ss} - {errorMessage}");
        if (errorLog.Count > 5)
            errorLog.Dequeue();

        Console.WriteLine($"\nОШИБКА СИСТЕМЫ: {errorMessage}");
    }

    public void PerformSystemCheck()
    {
        Console.WriteLine($"\n=== ПРОВЕРКА СИСТЕМЫ ({WarehouseName}) ===");
        Console.WriteLine($"Уровень автоматизации: {AutomationLevel}/5");
        Console.WriteLine($"Робототехника: {(HasRobotics ? "Работает" : "Отключена")}");
        Console.WriteLine($"Программное обеспечение: {SoftwareSystem}");
        Console.WriteLine($"Ошибок в логе: {errorLog.Count}");
    }
}

// GENERIC МЕНЕДЖЕР
public class InventoryManager<T> where T : Inventory
{
    private List<T> inventories;

    public InventoryManager()
    {
        inventories = new List<T>();
    }

    public void AddInventory(T inventory)
    {
        inventories.Add(inventory);
    }

    public void DisplayAllInventories()
    {
        Console.WriteLine($"\n=== ВСЕ СКЛАДЫ В МЕНЕДЖЕРЕ ({inventories.Count} шт.) ===");
        foreach (var inventory in inventories)
        {
            inventory.GetStorageStatus();
            Console.WriteLine();
        }
    }

    public List<T> GetInventoriesByCondition(Func<T, bool> condition)
    {
        return inventories.Where(condition).ToList();
    }

    public decimal CalculateTotalValue()
    {
        return inventories.Sum(inv => inv.CalculateTotalInventoryValue());
    }
}

// ОБРАБОТЧИКИ СОБЫТИЙ
public static void OnItemAdded(string message)
{
    Console.WriteLine($"    [СОБЫТИЕ] {message}");
}

public static void OnItemRemoved(string message)
{
    Console.WriteLine($"    [СОБЫТИЕ] {message}");
}

// === ОСНОВНОЙ КОД ПРОГРАММЫ ===
Console.WriteLine("🚀 ЗАПУСК СИСТЕМЫ УПРАВЛЕНИЯ СКЛАДАМИ");
Console.WriteLine("=====================================\n");

// СОЗДАНИЕ ТОВАРОВ
Console.WriteLine("1. СОЗДАНИЕ ТОВАРОВ");
var laptop = new Item("ITEM001", "Ноутбук Lenovo", 45000m, 10, 2.5, "Электроника", "ТехноПоставка");
var phone = new Item("ITEM002", "Смартфон Samsung", 25000m, 25, 0.3, "Электроника", "ТехноПоставка");
var chair = new Item("ITEM003", "Офисное кресло", 15000m, 5, 15.0, "Мебель", "ОфисМир");
var paper = new Item("ITEM004", "Бумага A4", 500m, 100, 5.0, "Канцелярия", "КанцТорг");
Console.WriteLine("✅ Создано 4 товара");
Console.WriteLine();

// СОЗДАНИЕ СКЛАДОВ
Console.WriteLine("2. СОЗДАНИЕ СКЛАДОВ");
var personal = new PersonalInventory("WH001", "Личный склад Петра", 1000, "Москва", "Иван Иванов", 
                                   "Петр Сидоров", "+7-999-123-45-67", true, 50);

var group = new GroupInventory("WH002", "Склад электроники", 5000, "Санкт-Петербург", "Анна Петрова",
                            "Электронные устройства", "Сергей Кузнецов", 1000000m);

var automated = new AutomatedInventory("WH003", "Автоматизированный склад", 10000, "Казань", "Мария Смирнова",
                                    4, true, "WarehousePro v3.0");

Console.WriteLine("✅ Создано 3 склада разных типов");
Console.WriteLine();

// ПОДПИСКА НА СОБЫТИЯ
Console.WriteLine("3. ПОДПИСКА НА СОБЫТИЯ");
personal.ItemAdded += OnItemAdded;
personal.ItemRemoved += OnItemRemoved;
group.ItemAdded += OnItemAdded;
group.ItemRemoved += OnItemRemoved;
automated.ItemAdded += OnItemAdded;
automated.ItemRemoved += OnItemRemoved;
Console.WriteLine("✅ Подписка на события выполнена");
Console.WriteLine();

// ДОБАВЛЕНИЕ ТОВАРОВ
Console.WriteLine("4. ДОБАВЛЕНИЕ ТОВАРОВ НА СКЛАДЫ");

Console.WriteLine("\n--- Персональный склад ---");
personal.AddItem(laptop);
personal.AddItem(paper);

Console.WriteLine("\n--- Групповой склад ---");
group.AddItem(laptop);
group.AddItem(phone);

Console.WriteLine("\n--- Автоматизированный склад ---");
automated.AddItem(laptop);
automated.AddItem(phone);
automated.AddItem(chair);
automated.AddItem(paper);
Console.WriteLine();

// ДЕМОНСТРАЦИЯ ПОЛИМОРФИЗМА
Console.WriteLine("5. ДЕМОНСТРАЦИЯ ПОЛИМОРФИЗМА");
Inventory[] allInventories = { personal, group, automated };
foreach (var inventory in allInventories)
{
    inventory.GetStorageStatus();
    Console.WriteLine();
}

// РАБОТА С КОЛЛЕКЦИЯМИ
Console.WriteLine("6. РАБОТА С КОЛЛЕКЦИЯМИ");
personal.DisplayRecentAdditions();
group.DisplayGroupCategories();
automated.PerformSystemCheck();
Console.WriteLine();

// GENERIC МЕНЕДЖЕР
Console.WriteLine("7. РАБОТА С GENERIC МЕНЕДЖЕРОМ");
var manager = new InventoryManager<Inventory>();
manager.AddInventory(personal);
manager.AddInventory(group);
manager.AddInventory(automated);

manager.DisplayAllInventories();

var largeWarehouses = manager.GetInventoriesByCondition(inv => inv.StorageCapacity > 2000);
Console.WriteLine($"📊 Склады с вместимостью > 2000 кг: {largeWarehouses.Count}");

var totalValue = manager.CalculateTotalValue();
Console.WriteLine($"💰 Общая стоимость всех товаров: {totalValue:C}");
Console.WriteLine();

// ДОПОЛНИТЕЛЬНЫЕ ОПЕРАЦИИ
Console.WriteLine("8. ДОПОЛНИТЕЛЬНЫЕ ОПЕРАЦИИ");
personal.ContactOwner();
group.AllocateBudget(50000m);
automated.LogError("Незначительный сбой в системе конвейера");

Console.WriteLine("\n--- Поиск электроники ---");
var electronics = group.FindItemsByCategory("Электроника");
Console.WriteLine($"🔍 Найдено товаров в категории 'Электроника': {electronics.Count}");

Console.WriteLine("\n--- Статистика по категориям ---");
personal.DisplayCategoryStatistics();
group.DisplayCategoryStatistics();
automated.DisplayCategoryStatistics();
Console.WriteLine();

// ТЕСТИРОВАНИЕ УДАЛЕНИЯ
Console.WriteLine("9. ТЕСТИРОВАНИЕ УДАЛЕНИЯ ТОВАРОВ");
personal.RemoveItem(laptop);
group.RemoveItem(phone);
Console.WriteLine();

// ФИНАЛЬНЫЙ ОТЧЕТ
Console.WriteLine("10. ФИНАЛЬНЫЙ ОТЧЕТ");
foreach (var inventory in allInventories)
{
    inventory.DisplayAllItems();
    Console.WriteLine($"💰 Общая стоимость товаров: {inventory.CalculateTotalInventoryValue():C}");
    Console.WriteLine();
}

Console.WriteLine("=====================================");
Console.WriteLine("✅ ПРОГРАММА УСПЕШНО ЗАВЕРШЕНА!");

🚀 ЗАПУСК СИСТЕМЫ УПРАВЛЕНИЯ СКЛАДАМИ

1. СОЗДАНИЕ ТОВАРОВ
✅ Создано 4 товара

2. СОЗДАНИЕ СКЛАДОВ
✅ Создано 3 склада разных типов

3. ПОДПИСКА НА СОБЫТИЯ
✅ Подписка на события выполнена

4. ДОБАВЛЕНИЕ ТОВАРОВ НА СКЛАДЫ

--- Персональный склад ---
    [СОБЫТИЕ] Ноутбук Lenovo добавлен на склад Личный склад Петра
    УСПЕХ: Ноутбук Lenovo добавлен на склад
    [СОБЫТИЕ] Бумага A4 добавлен на склад Личный склад Петра
    УСПЕХ: Бумага A4 добавлен на склад

--- Групповой склад ---
    [СОБЫТИЕ] Ноутбук Lenovo добавлен на склад Склад электроники
    УСПЕХ: Ноутбук Lenovo добавлен на склад
    Добавлен в группу 'Электронные устройства'
    [СОБЫТИЕ] Смартфон Samsung добавлен на склад Склад электроники
    УСПЕХ: Смартфон Samsung добавлен на склад
    Добавлен в группу 'Электронные устройства'

--- Автоматизированный склад ---
    [СОБЫТИЕ] Ноутбук Lenovo добавлен на склад Автоматизированный склад
    УСПЕХ: Ноутбук Lenovo добавлен на склад
    [СОБЫТИЕ] Смартфон Samsung добавлен на склад Авт